In [1]:
# -*- coding: utf-8 -*-
"""Thí nghiệm 1 (phương án A): P0 proxy + ROC/PR trên Val-select + Pre-Test audit.

Chỉ sử dụng Train và Val-select để tạo/đánh giá Proxy.
Chỉ đọc các kết quả đã LƯU của Val-calibrate để xác nhận Q99.
KHÔNG chấm Test, KHÔNG huấn luyện lại và KHÔNG chỉnh ngưỡng Q99.

Chạy tại thư mục Stocks/notebooks/ (hoặc bên cạnh các tệp CSV đã xuất):
    python 07_P0_finalize_pretest_A.py
"""
from __future__ import annotations
import hashlib
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, auc

# Tự xác định thư mục gốc du an: tim tu dong ca khi VS Code mo workspace o cap cha.
# Neu can, sua ROOT_OVERRIDE thanh Path(r"D:\LUẬN VĂN\stock-news-anomaly-project\Stocks\notebooks")
ROOT_OVERRIDE = None

def choose_root() -> Path:
    if ROOT_OVERRIDE is not None:
        root = Path(ROOT_OVERRIDE).expanduser().resolve()
        if not ((root / "outputs/03_feature_engineering/03_model_ready_development.csv").is_file()
                or (root / "03_model_ready_development.csv").is_file()):
            raise FileNotFoundError(f"ROOT_OVERRIDE không trỏ đến nơi có dữ liệu Bước 03: {root}")
        return root
    roots = []
    cwd = Path.cwd().resolve()
    for parent in [cwd, *list(cwd.parents)[:3]]:
        roots.extend([parent, parent / "notebooks", parent / "Stocks" / "notebooks"])
    if "__file__" in globals():
        roots.insert(0, Path(__file__).resolve().parent)
    visited = set()
    for root in roots:
        if root in visited:
            continue
        visited.add(root)
        if (root / "outputs/03_feature_engineering/03_model_ready_development.csv").is_file() or (root / "03_model_ready_development.csv").is_file():
            return root
    raise FileNotFoundError(
        "Không tìm được thư mục Stocks/notebooks có outputs/03_feature_engineering/03_model_ready_development.csv. "
        "Hãy đặt ROOT_OVERRIDE ở đầu ô cấu hình."
    )

ROOT = choose_root()
print("Thư mục gốc đã xác định:", ROOT)

OUTPUT = ROOT / 'outputs' / '07_P0_final_pretest_A'
OUTPUT.mkdir(parents=True, exist_ok=True)
FEATURES = ['z_gap', 'z_intraday', 'z_range', 'z_volume']
MODELS = ['IF', 'AE', 'LOF']
TOLERANCES = {'IF': 1e-5, 'AE': 5e-4, 'LOF': 1e-5}
P0_PRICE_K = 2.0
P0_VOLUME_K = 3.0
P0_WINDOW = 20
BOOTSTRAP_REPS = 500
BOOTSTRAP_SEED = 42


def find_file(*candidates: str) -> Path:
    for candidate in candidates:
        p = ROOT / candidate
        if p.is_file():
            return p
    raise FileNotFoundError('Cannot find any of: ' + ', '.join(candidates))


def sha256_file(path: Path) -> str:
    sha = hashlib.sha256()
    with path.open('rb') as fh:
        for block in iter(lambda: fh.read(1_048_576), b''):
            sha.update(block)
    return sha.hexdigest()


path = {
    'features': find_file('outputs/03_feature_engineering/03_features_all.csv','03_features_all.csv'),
    'dev': find_file('outputs/03_feature_engineering/03_model_ready_development.csv','03_model_ready_development.csv'),
    'feature_config': find_file('outputs/03_feature_engineering/03_feature_config.json','03_feature_config.json'),
    'scores': find_file('outputs/06_5_model_comparison_A/06_5_aligned_val_scores.csv','06_5_aligned_val_scores.csv'),
    'thresholds': find_file('outputs/07_threshold_calibration_A/07_thresholds.json','07_thresholds.json'),
    'cal': find_file('outputs/07_threshold_calibration_A/07_calibration_scores.csv','07_calibration_scores.csv'),
    'IF': find_file('outputs/05_model_training_A/05_if.joblib','05_if.joblib'),
    'AE': find_file('outputs/06_2_ae_validation_A/06_2_ae_seed42_lr001_epoch1495_best.pt','06_2_ae_seed42_lr001_epoch1495_best.pt'),
    'LOF': find_file('outputs/06_3_lof_validation_A/06_3_lof_baseline_k20_euclidean.joblib','06_3_lof_baseline_k20_euclidean.joblib'),
}

Thư mục gốc đã xác định: D:\LUẬN VĂN\stock-news-anomaly-project\Stocks\notebooks


In [2]:
# 1. Verify that saved model/feature files are exactly those in Step07 threshold manifest.
manifest = json.loads(path['thresholds'].read_text(encoding='utf-8'))
config = json.loads(path['feature_config'].read_text(encoding='utf-8'))
assert manifest['quantile'] == 0.99 and manifest['numpy_quantile_method'] == 'linear'
assert manifest['rule'] == 'score > threshold'
assert manifest['feature_cols'] == FEATURES
assert config['features_for_model'] == FEATURES
assert config['window_prior_valid_feature_values_per_ticker'] == 60
assert sha256_file(path['features']) == manifest['feature_data_sha256']
assert sha256_file(path['feature_config']) == manifest['feature_config_sha256']
for model in MODELS:
    assert sha256_file(path[model]) == manifest['model_sha256'][model], f'Model checksum changed: {model}'

In [3]:
# 2. Load only TRAIN and VAL-SELECT fields (no Test rows enter the proxy computation).
cols = ['ticker','date','close','volume','split','val_role','is_model_ready','prev_valid_close']
parts = []
for chunk in pd.read_csv(path['features'], usecols=cols, chunksize=4000):
    allowed = chunk['split'].eq('train') | (chunk['split'].eq('val') & chunk['val_role'].eq('select'))
    parts.append(chunk.loc[allowed].copy())
raw = pd.concat(parts, ignore_index=True)
raw['date'] = pd.to_datetime(raw['date'], format='%Y-%m-%d', errors='raise')
raw = raw.sort_values(['ticker','date']).reset_index(drop=True)
assert raw['split'].isin(['train','val']).all()
assert len(raw) == 11180 + 1430, len(raw)
assert raw['ticker'].nunique() == 13
assert not raw.duplicated(['ticker','date']).any()
assert (raw['close'] > 0).all() and (raw['volume'] >= 0).all()
assert raw.loc[raw.split.eq('val'),'val_role'].eq('select').all()

In [4]:
# 3. Approved P0 from older experiment description, applied to current OHLCV Close.
#    Rolling history is STRICTLY PRIOR (shift(1)), ticker-specific.
by = raw.groupby('ticker', sort=False)
raw['return_cc'] = np.log(raw['close'] / by['close'].shift(1))
raw['return_std_prior20'] = raw.groupby('ticker')['return_cc'].transform(
    lambda x: x.shift(1).rolling(P0_WINDOW, min_periods=P0_WINDOW).std(ddof=1)
)
raw['volume_mean_prior20'] = raw.groupby('ticker')['volume'].transform(
    lambda x: x.shift(1).rolling(P0_WINDOW, min_periods=P0_WINDOW).mean()
)
raw['proxy_eligible'] = (
    raw['return_cc'].notna()
    & raw['return_std_prior20'].notna()
    & (raw['return_std_prior20'] > 0)
    & raw['volume_mean_prior20'].notna()
    & (raw['volume_mean_prior20'] > 0)
)
raw['proxy_price'] = raw['return_cc'].abs() > P0_PRICE_K * raw['return_std_prior20']
raw['proxy_volume'] = raw['volume'] > P0_VOLUME_K * raw['volume_mean_prior20']
raw['proxy_joint'] = raw['proxy_price'] & raw['proxy_volume']
raw['proxy_label'] = np.where(raw['proxy_eligible'], (raw['proxy_price'] | raw['proxy_volume']).astype(int), -1)

# Validate last historical close matches the Step03 definition for all model-ready samples.
raw['independent_prev_close'] = by['close'].shift(1)
ready = raw.loc[raw['is_model_ready']].copy()
assert len(ready) == 10387 + 1430
assert ready['proxy_eligible'].all(), 'Some model-ready rows cannot be given P0 label'
assert np.allclose(ready['independent_prev_close'], ready['prev_valid_close'], rtol=0, atol=1e-8)
train = ready.loc[ready['split'].eq('train')]
select = ready.loc[ready['split'].eq('val') & ready['val_role'].eq('select')]
assert len(train) == 10387 and len(select) == 1430
assert train.date.max() < select.date.min()

In [5]:
# 4. Merge the SAME Val-select sessions scored by all three existing models.
dev = pd.read_csv(path['dev'])
assert len(dev.loc[dev['split'].eq('train')]) == 10387
assert len(dev.loc[dev['split'].eq('val') & dev['val_role'].eq('select')]) == 1430
assert not dev['split'].eq('test').any()
base_dev = dev.loc[dev.split.eq('val') & dev.val_role.eq('select'), ['ticker','date'] + FEATURES].copy()
base_dev['date'] = pd.to_datetime(base_dev['date'],format='%Y-%m-%d')
scores = pd.read_csv(path['scores'])
scores['date'] = pd.to_datetime(scores['date'], format='%Y-%m-%d', errors='raise')
assert len(scores) == 1430 and scores.ticker.nunique() == 13 and scores.date.nunique() == 110
assert not scores.duplicated(['ticker','date']).any()
sel = select.merge(scores[['ticker','date','score_if','score_ae','score_lof']],
                   on=['ticker','date'], how='inner', validate='one_to_one')
assert len(sel) == len(select) == len(scores) == 1430
cross = sel.merge(base_dev, on=['ticker','date'], validate='one_to_one')
assert len(cross) == 1430
assert np.isfinite(cross[FEATURES].to_numpy(dtype=float)).all()
assert (cross.proxy_label >= 0).all()
for model in MODELS:
    assert np.isfinite(sel[f'score_{model.lower()}']).all()

In [6]:
# 5. Reproduce model score on the select samples, without training, to guard checkpoint mismatch.
class SmallAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(4,2),nn.Tanh(),nn.Linear(2,4))
    def forward(self,x):
        return self.net(x)

torch.set_num_threads(1)
with warnings.catch_warnings(record=True) as detected:
    warnings.simplefilter('always')
    model_if = joblib.load(path['IF'])
    model_lof = joblib.load(path['LOF'])
assert isinstance(model_if, IsolationForest)
assert isinstance(model_lof, LocalOutlierFactor)
assert model_if.n_estimators == 200 and model_if.max_samples == 256
assert model_lof.n_neighbors == 20 and model_lof.novelty
ckpt = torch.load(path['AE'], map_location='cpu', weights_only=True)
assert ckpt['best_epoch'] == 1495 and list(ckpt['features']) == FEATURES
model_ae = SmallAE()
model_ae.load_state_dict(ckpt['state_dict'],strict=True)
model_ae.eval()
sel = sel.sort_values(['date','ticker']).reset_index(drop=True)
X = np.array(sel.merge(base_dev,on=['ticker','date'],validate='one_to_one')[FEATURES], dtype=np.float32,copy=True)
with torch.no_grad():
    t = torch.from_numpy(X)
    ae_scores = (model_ae(t)-t).square().mean(dim=1).cpu().numpy().astype(float)
recomputed = {
    'IF': -model_if.score_samples(X),
    'AE': ae_scores,
    'LOF': -model_lof.score_samples(X),
}
model_diff = {}
for model in MODELS:
    err = float(np.max(np.abs(recomputed[model] - sel[f'score_{model.lower()}'].to_numpy(float))))
    model_diff[model] = err
    assert err < TOLERANCES[model], f'Checkpoint score mismatch for {model}: {err}'

In [7]:
# 6. ROC-AUC, average precision (AP) and trapezoidal PR-AUC on Val-select ONLY.
label = sel['proxy_label'].to_numpy(int)
assert np.unique(label).tolist() == [0,1]
prevalence = float(label.mean())
metric_rows=[]
for model in MODELS:
    s = sel[f'score_{model.lower()}'].to_numpy(float)
    precision, recall, _ = precision_recall_curve(label, s)
    auc_roc = float(roc_auc_score(label,s))
    avg_precision = float(average_precision_score(label,s))
    pr_trapz = float(auc(recall, precision))
    threshold_sel = float(np.quantile(s,0.99,method='linear'))
    flags = s > threshold_sel
    metric_rows.append({
        'model':model,
        'proxy':'P0_final_std20_mean20_OR',
        'n_val_select':len(sel),
        'proxy_positives':int(label.sum()),
        'proxy_prevalence':prevalence,
        'roc_auc_proxy':auc_roc,
        'average_precision_proxy':avg_precision,
        'pr_auc_trapezoid_proxy':pr_trapz,
        'val_select_q99_preview_only':threshold_sel,
        'n_val_select_top1':int(flags.sum()),
        'n_proxy_in_top1':int(np.sum(flags & (label==1))),
        'max_baseline_score_diff':model_diff[model],
    })
metrics = pd.DataFrame(metric_rows)

In [8]:
# 7. Exploratory date-cluster bootstrap intervals; 13 same-day tickers are correlated.
rng = np.random.default_rng(BOOTSTRAP_SEED)
unique_days = np.array(sorted(sel.date.unique()))
indices_by_day = [np.flatnonzero(sel.date.to_numpy()==day) for day in unique_days]
boot_rows=[]
for _ in range(BOOTSTRAP_REPS):
    draws = rng.integers(0,len(indices_by_day),len(indices_by_day))
    rows = np.concatenate([indices_by_day[i] for i in draws])
    yt = label[rows]
    if np.unique(yt).size < 2:
        continue
    for model in MODELS:
        st = sel[f'score_{model.lower()}'].to_numpy(float)[rows]
        boot_rows.append({'model':model,'auc':roc_auc_score(yt,st),'ap':average_precision_score(yt,st)})
boot = pd.DataFrame(boot_rows)
for row in metric_rows:
    bd = boot.loc[boot.model.eq(row['model'])]
    row['roc_auc_ci_lower'] = float(bd['auc'].quantile(.025))
    row['roc_auc_ci_upper'] = float(bd['auc'].quantile(.975))
    row['ap_ci_lower'] = float(bd['ap'].quantile(.025))
    row['ap_ci_upper'] = float(bd['ap'].quantile(.975))
metrics = pd.DataFrame(metric_rows)

In [9]:
# 8. QA Calibration — no calibration labels/proxies created here; no tuning.
cal = pd.read_csv(path['cal'])
assert len(cal)==manifest['n_calibration']==962
assert cal.ticker.nunique()==13 and cal.date.nunique()==74
assert cal.date.min()==manifest['calibration_first_date']
assert cal.date.max()==manifest['calibration_last_date']
assert select.date.max().strftime('%Y-%m-%d') < cal.date.min()
threshold_qas={}
for model in MODELS:
    c = f'score_{model.lower()}'
    score = cal[c].to_numpy(float)
    t = manifest['thresholds'][model]['threshold']
    rep = float(np.quantile(score, .99, method='linear'))
    assert abs(rep-t) < 1e-10
    pred = (score>t).astype(int)
    assert np.array_equal(pred,cal[f'{model.lower()}_prediction'].to_numpy(int))
    assert pred.sum()==manifest['thresholds'][model]['n_calibration_alerts']==10
    threshold_qas[model]={'threshold':t,'n_calibrate_alerts':int(pred.sum())}

In [10]:
# 9. OUTPUTS. No Test data, scoring, refit, recalibration, or Proxy-based model selection.
metric_cols=['ticker','date','return_cc','return_std_prior20','volume','volume_mean_prior20',
             'proxy_eligible','proxy_price','proxy_volume','proxy_joint','proxy_label',
             'score_if','score_ae','score_lof']
sel[metric_cols].assign(date=lambda x:x.date.dt.strftime('%Y-%m-%d')).to_csv(
    OUTPUT/'07_p0_val_select_scores.csv',index=False,encoding='utf-8-sig')
metrics.to_csv(OUTPUT/'07_p0_roc_pr_val_select.csv',index=False,encoding='utf-8-sig')

by_split = ready.groupby(['split','val_role'],sort=True).agg(
    n=('proxy_label','size'), proxy_positives=('proxy_label','sum'),
    price_positives=('proxy_price','sum'), volume_positives=('proxy_volume','sum'),
    price_and_volume=('proxy_joint','sum')
).reset_index()
by_split['proxy_prevalence_pct'] = (by_split.proxy_positives/by_split.n)*100
by_split.to_csv(OUTPUT/'07_p0_by_split.csv',index=False,encoding='utf-8-sig')

proxy_definition = {
    'name':'P0_final_std20_mean20_OR', 'status':'APPROVED_IN_CONVERSATION',
    'source_input':['ticker','date','open','high','low','close','volume'],
    'used_ohlcv':['ticker','date','close','volume'],
    'historical_window':20,
    'return':'ln(close_t / prior_valid_close_t)',
    'past_volatility':'std(last 20 prior log close-to-close returns, ddof=1)',
    'price_rule':'abs(return_t) > 2 * past_volatility',
    'past_mean_volume':'mean(last 20 prior eligible session volumes)',
    'volume_rule':'volume_t > 3 * past_mean_volume',
    'combine':'price_rule OR volume_rule',
    'missing_history':'not_eligible, never cast missing to proxy=0',
    'valid_data_scope':'Train + Val-select only',
    'proxy_intended_meaning':'large absolute close-close move OR unusual volume surge',
    'proxy_not_ground_truth':True,
    'proxy_not_used_for_model_fit_or_threshold':True,
    'does_not_cover':['intraday reversal with little close-close move','range-only anomaly','volume collapse'],
}
(OUTPUT/'07_p0_proxy_definition.json').write_text(json.dumps(proxy_definition,ensure_ascii=False,indent=2),encoding='utf-8')

checklist = {
    'data_03_ready_and_chronological':True,
    'train_n':int(len(train)),
    'val_select_n':int(len(sel)),
    'proxy_valid_all_model_ready_rows':bool(ready.proxy_eligible.all()),
    'proxy_past_only':True,
    'roc_pr_on_val_select_only':True,
    'checkpoint_files_match_step07_hash':True,
    'checkpoint_scores_reproduce_step065':True,
    'step07_q99_reproduced_without_refit':True,
    'test_scores_computed':False,
    'test_used_to_choose_model_or_proxy':False,
    'q99_unchanged':True,
    'proxy_policy_awaiting_user_agreement':False,
    'test_open_authorized':True,
    'recommendation':'Technical gate passed. Test is ready for a separate one-time evaluation; this script does NOT score Test.'
}
(OUTPUT/'07_pretest_gate.json').write_text(json.dumps(checklist,ensure_ascii=False,indent=2),encoding='utf-8')

report = [
    '# Thí nghiệm 1 — Proxy P0 và kiểm tra trước Test (chỉ dữ liệu phát triển)',
    '',
    '## Quy tắc P0 đã thống nhất',
    '- `price=1` khi `abs(log(C_t/C_prev)) > 2*Std(previous 20 log returns, ddof=1)`.',
    '- `volume=1` khi `V_t > 3*Mean(previous 20 volumes)`.',
    '- `P0=price OR volume`; lịch sử từng mã, shift(1), không dùng tương lai.',
    '- P0 không phải ground truth; không bao quát Range-only, intraday reversal và volume collapse.',
    '', '## Số liệu Proxy',
    '```',by_split.to_string(index=False), '```',
    '', '## ROC-AUC / AP / PR-AUC (chỉ Val-select)',
    '```',metrics.round(6).to_string(index=False), '```',
    '', '## QA trước Test',
    f'- SHA-256 checkpoint IF/AE/LOF và Feature-config khớp Step07; chấm lại Val-select sai khác tối đa: {model_diff}.',
    f'- Q99 Val-cal giữ nguyên; số nhãn 1 mỗi mô hình: {threshold_qas}.',
    '- Validation-select đã được dùng trong phát triển mô hình; AUC/AP là kết quả phát triển, không phải Test độc lập.',
    '- Chưa chấm Test, chưa huấn luyện lại, chưa điều chỉnh mô hình hay ngưỡng.',
    '- Quy tắc P0 đã được đồng ý triển khai; khóa manifest trước khi mở Test.',
    '- Không dùng AUC/AP so với P0 để tuyên bố độ chính xác thật hoặc thay Q99.',
]
(OUTPUT/'07_proxy_pretest_report.md').write_text('\n'.join(report),encoding='utf-8')

print('PASS: complete P0 proxy and pretest audit WITHOUT Test scoring')
print('Files in', OUTPUT.resolve())
print(by_split.to_string(index=False))
print(metrics[['model','roc_auc_proxy','average_precision_proxy','pr_auc_trapezoid_proxy',
               'roc_auc_ci_lower','roc_auc_ci_upper']].to_string(index=False))
print('Checkpoint validation max abs diffs:',model_diff)
print('Technical Test readiness: PASS; this script did NOT score Test')

PASS: complete P0 proxy and pretest audit WITHOUT Test scoring
Files in D:\LUẬN VĂN\stock-news-anomaly-project\Stocks\notebooks\outputs\07_P0_final_pretest_A
split       val_role     n  proxy_positives  price_positives  volume_positives  price_and_volume  proxy_prevalence_pct
train not_applicable 10387              845              753               223               131              8.135169
  val         select  1430              149              137                28                16             10.419580
model  roc_auc_proxy  average_precision_proxy  pr_auc_trapezoid_proxy  roc_auc_ci_lower  roc_auc_ci_upper
   IF       0.922895                 0.556061                0.553233          0.892513          0.946022
   AE       0.898218                 0.534810                0.532950          0.859017          0.926599
  LOF       0.794199                 0.490199                0.488863          0.713811          0.852326
Checkpoint validation max abs diffs: {'IF': 0.0, 'AE': 0.0, '

In [11]:
# 10. Freeze the actual policy used in Step08 without touching Test records.
freeze = {
    "experiment": "EXP1_A_P0",
    "status": "READY_FOR_SEPARATE_TEST_EVALUATION",
    "approval": "P0 accepted by user in current conversation",
    "source_columns": ["ticker", "date", "open", "high", "low", "close", "volume"],
    "model_features": FEATURES,
    "proxy_name": "P0_final_std20_mean20_OR",
    "proxy_return": "ln(Close_t / previous_valid_Close)",
    "proxy_price": "abs(return_t) > 2 * sample_std(previous_20_returns, ddof=1)",
    "proxy_volume": "Volume_t > 3 * mean(previous_20_volumes)",
    "proxy_combination": "OR",
    "proxy_history": "ticker-specific, shift(1), no current/future observations",
    "proxy_missing_history": "not eligible, not 0",
    "proxy_not_ground_truth": True,
    "proxy_not_for_fitting_or_threshold_selection": True,
    "calibration_quantile": 0.99,
    "threshold_method": "numpy_linear_percentile, score strictly > threshold",
    "thresholds": {m: float(manifest["thresholds"][m]["threshold"]) for m in MODELS},
    "feature_window": 60,
    "feature_config_sha256": sha256_file(path["feature_config"]),
    "feature_data_sha256": sha256_file(path["features"]),
    "development_data_sha256": sha256_file(path["dev"]),
    "val_select_score_sha256": sha256_file(path["scores"]),
    "step07_threshold_manifest_sha256": sha256_file(path["thresholds"]),
    "model_sha256": {m: sha256_file(path[m]) for m in MODELS},
    "n_train_model_ready": int(len(train)),
    "n_val_select": int(len(sel)),
    "n_val_calibrate": int(len(cal)),
    "n_val_select_proxy_positive": int(label.sum()),
    "test_scored": False,
    "models_refit": False,
    "test_policy": "Run once after this gate, score fixed models and rule on Test; no refit/recalibration",
}
(OUTPUT / "07_A_P0_frozen_policy.json").write_text(
    json.dumps(freeze, ensure_ascii=False, indent=2), encoding="utf-8"
)
# Read back official outputs; fail loudly if they do not match.
qamt = pd.read_csv(OUTPUT / "07_p0_roc_pr_val_select.csv")
qap0 = pd.read_csv(OUTPUT / "07_p0_val_select_scores.csv")
qafz = json.loads((OUTPUT / "07_A_P0_frozen_policy.json").read_text(encoding="utf-8"))
assert set(qamt.model) == set(MODELS) and len(qamt) == 3
assert len(qap0) == 1430 and int(qap0.proxy_label.sum()) == int(label.sum())
assert int(label.sum()) == 149 and qafz["test_scored"] is False
assert qafz["thresholds"]["IF"] == manifest["thresholds"]["IF"]["threshold"]
print("FROZEN CONFIG:", (OUTPUT / "07_A_P0_frozen_policy.json").resolve())

FROZEN CONFIG: D:\LUẬN VĂN\stock-news-anomaly-project\Stocks\notebooks\outputs\07_P0_final_pretest_A\07_A_P0_frozen_policy.json


In [12]:
# 11. Four-case comparison ON VAL-SELECT, using a local Q99 for exploration ONLY.
# This is NOT the official Q99 calibrated on Val-calibrate, and does not select parameters.
quadrants=[]
for model in MODELS:
    local_q99=float(np.quantile(sel[f"score_{model.lower()}"].to_numpy(float),0.99,method="linear"))
    flagged=(sel[f"score_{model.lower()}"].to_numpy(float)>local_q99).astype(int)
    for m,p in [(1,1),(1,0),(0,1),(0,0)]:
        quadrants.append({
            "model":model,"diagnostic_only":True,
            "threshold_source":"Val-select_Q99_not_official",
            "model_flag":m,"proxy_P0":p,
            "n_sessions":int(np.sum((flagged==m)&(label==p))),
        })
quadrants=pd.DataFrame(quadrants)
assert quadrants.groupby('model')['n_sessions'].sum().eq(len(sel)).all()
quadrants.to_csv(OUTPUT/'07_p0_val_select_four_cases_diagnostic.csv',index=False,encoding='utf-8-sig')

In [13]:
# 12. ROC and Precision-Recall figures from actual model scores (development only).
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
fig,ax=plt.subplots(figsize=(7.8,5.3))
for model in MODELS:
    score=sel[f"score_{model.lower()}"].to_numpy(float)
    fpr,tpr,_=roc_curve(label,score)
    aucv=float(metrics.loc[metrics.model.eq(model),'roc_auc_proxy'].iloc[0])
    ax.plot(fpr,tpr,label=f"{model} (AUC={aucv:.3f})")
ax.plot([0,1],[0,1],'--',color='gray',linewidth=1,label='Random ranking')
ax.set(xlabel='False positive rate vs Proxy P0',ylabel='True positive rate vs Proxy P0',title='Val-select — ROC vs P0 (not ground truth)')
ax.legend();ax.grid(alpha=.25);fig.tight_layout()
fig.savefig(OUTPUT/'07_p0_roc_val_select.png',dpi=150)
plt.close(fig)

fig,ax=plt.subplots(figsize=(7.8,5.3))
for model in MODELS:
    score=sel[f"score_{model.lower()}"].to_numpy(float)
    precision,recall,_=precision_recall_curve(label,score)
    ap=float(metrics.loc[metrics.model.eq(model),'average_precision_proxy'].iloc[0])
    ax.plot(recall,precision,label=f"{model} (AP={ap:.3f})")
ax.axhline(prevalence,ls='--',color='gray',lw=1,label=f'P0 prevalence ({prevalence:.1%})')
ax.set(xlabel='Recall vs Proxy P0',ylabel='Precision vs Proxy P0',title='Val-select — PR vs P0 (not ground truth)')
ax.legend();ax.grid(alpha=.25);fig.tight_layout()
fig.savefig(OUTPUT/'07_p0_pr_val_select.png',dpi=150)
plt.close(fig)
print('Four-case diagnostic (Val-select only):')
print(quadrants.pivot_table(index='model',columns=['model_flag','proxy_P0'],values='n_sessions').to_string())
print('READY: P0 frozen; existing Q99 unchanged; no Test evaluated.')

Four-case diagnostic (Val-select only):
model_flag       0           1      
proxy_P0         0      1    0     1
model                               
AE          1279.0  136.0  2.0  13.0
IF          1276.0  139.0  5.0  10.0
LOF         1280.0  135.0  1.0  14.0
READY: P0 frozen; existing Q99 unchanged; no Test evaluated.


In [14]:
# 13. Independent spot-checks of strict past-only P0 history, one per ticker.
for ticker, group in sel.groupby("ticker",sort=True):
    row=group.sort_values("date").iloc[0]
    history=raw.loc[(raw.ticker==ticker)&(raw.date<row.date)].sort_values("date")
    past_returns=history["return_cc"].dropna().tail(20).to_numpy(float)
    past_volumes=history["volume"].tail(20).to_numpy(float)
    assert len(past_returns)==20 and len(past_volumes)==20
    assert np.isclose(float(np.std(past_returns,ddof=1)),float(row.return_std_prior20),rtol=1e-12,atol=1e-12)
    assert np.isclose(float(np.mean(past_volumes)),float(row.volume_mean_prior20),rtol=1e-12,atol=1e-12)
print('PAST-ONLY QA: PASS for first Val-select row of all',sel.ticker.nunique(),'tickers')

PAST-ONLY QA: PASS for first Val-select row of all 13 tickers
